# CellPert — End-to-End Tutorial

This notebook walks through a full CellPert use case from a clean checkout:

1. **Environment & data setup** — install dependencies and pull the LINCS / Tahoe datasets and pretrained weights from HuggingFace.
2. **Load the pretrained GINVAE model.**
3. **Build the reference biological context** from LINCS (control vs. perturb latent centroids).
4. **Predict perturbation responses** on a Tahoe query plate.
5. **Evaluate** the prediction (Pearson / Spearman / DEG-delta) on paired control–perturb samples.
All paths are relative to the repo root (`CellPert/`). After downloading the data the layout should look like:

```
CellPert/
├── src/
│   └── best_gin_vae_model_node_level.pth
├── data/
│   ├── lincs/merged_all_965_with_morgan.h5ad
│   └── minitahoe/p1_with_morgan.h5ad ... p14_with_morgan.h5ad
```

## 1. Environment & data

Run these once in a shell — *not* inside the notebook unless you want to (un)comment the `!`-prefixed lines.

```bash
conda env create -f environment.yml
conda activate info
```

Then download the model weights and datasets from HuggingFace (the repo is public, no token needed):

In [1]:
# One-time download — skip if you already have the files locally.
import os, zipfile, tarfile
from huggingface_hub import hf_hub_download

REPO_ID = 'Mike2481/CellPert'

os.makedirs('./data', exist_ok=True)

# Pretrained checkpoint
hf_hub_download(
    repo_id=REPO_ID, filename='best_gin_vae_model_node_level.pth',
    repo_type='dataset', local_dir='./src',
)

# LINCS reference
lincs_zip = hf_hub_download(
    repo_id=REPO_ID, filename='lincs.zip',
    repo_type='dataset', local_dir='./data',
)
with zipfile.ZipFile(lincs_zip) as z:
    z.extractall('./data/')

# Tahoe query plates (tar.gz)
tahoe_tar = hf_hub_download(
    repo_id=REPO_ID, filename='minitahoe.tar.gz',
    repo_type='dataset', local_dir='./data',
)
with tarfile.open(tahoe_tar, 'r:gz') as t:
    t.extractall('./data/')

lincs.zip:   0%|          | 0.00/1.57G [00:00<?, ?B/s]

minitahoe.tar.gz:   0%|          | 0.00/341M [00:00<?, ?B/s]

## 2. Load the pretrained GINVAE

`GINVAE` defaults: `input_dim=1, hidden_dim=300, latent_dim=100, num_layers=2`. Match these to the checkpoint.

In [2]:
import sys
sys.path.insert(0, './src')

import torch
from model import GINVAE

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GINVAE(input_dim=1, hidden_dim=300, latent_dim=100, num_layers=2).to(device)
model.load_state_dict(torch.load('./src/best_gin_vae_model_node_level.pth', map_location=device))
model.eval();

/blue/qsong1/wang.qing/miniconda3/envs/info/lib/python3.11/site-packages/torch_geometric/typing.py:97: UserWarning: An issue occurred while importing 'torch-cluster'. Disabling its usage. Stacktrace: Could not load this library: /blue/qsong1/wang.qing/miniconda3/envs/info/lib/python3.11/site-packages/torch_cluster/_version_cuda.so
  warnings.warn(f"An issue occurred while importing 'torch-cluster'. "


## 3. Build the reference biological context

CellPert transfers a perturbation effect from a *reference* dataset (LINCS) to a *query* dataset (Tahoe). The reference contributes two centroids in latent space:

$$z^{\text{perturb}}_{\text{query}} = z^{\text{ctrl}}_{\text{query}} + (\bar z^{\text{perturb}}_{\text{ref}} - \bar z^{\text{ctrl}}_{\text{ref}})$$

The first call processes LINCS and caches `latent_ctrl_ref` / `latent_ptrb_ref` to disk; subsequent calls are instant.

In [3]:
from dataset import ChunkedGeneGraphDataset
import os

os.makedirs('./output', exist_ok=True)

lincs_path = './data/lincs/merged_all_965_with_morgan.h5ad'
lincs_dataset = ChunkedGeneGraphDataset(
    h5ad_paths=[lincs_path], split='train',
    chunk_size=10000, auto_build_graph=True,
    species=9606, required_score=700,
)

latent_ctrl_ref, latent_ptrb_ref, info = model.process_reference_dataset(
    x_ref_dataset=lincs_dataset,
    save_path='./output/reference_latents.pkl',
    force_reprocess=False,
)
print('ctrl latent shape :', latent_ctrl_ref.shape)
print('perturb latent shape:', latent_ptrb_ref.shape)

Found 16 chunks with 157138 total samples
Processing reference dataset...
Processing and saving reference dataset with 157138 samples...
Using batch size: 100, saving every 100 batches


Processing reference batches:   6%|▋         | 99/1572 [00:08<01:39, 14.82it/s, samples=10000-10099, processed=1e+4]            

Saved chunk with 10000 samples


Processing reference batches:  13%|█▎        | 199/1572 [00:15<01:33, 14.66it/s, samples=19900-19999, processed=19900]

Saved chunk with 10000 samples


...

Processing completed. Total samples processed: 157138
Reference data metadata saved to ./output/reference_latents.pkl
Data chunks saved in ./output/reference_latents_chunks
Total samples processed: 157138

Summary statistics:
  condition:
    control: 78569
    perturb: 78569
  celltype:
    1HAE: 646
    22RV1: 1230
    5637: 6
    A204: 1234
    A375: 7118
    A549: 10144
    AGS: 98
    AN3CA: 6
    ASC: 4954
    BC3C: 1152
    BEN: 1248
    BICR6: 174
    BJAB: 146
    BT20: 272
    BT474: 74
    C42: 10
    CAL29: 1250
    CD34: 352
    CJM: 1240
    COV434: 172
    CW2: 6
    DU145: 8
    DV90: 6
    ES2: 166
    G401: 178
    GI1: 1226
    GP2D: 168
    H1975: 20
    HA1E: 8010
    HBL1: 180
    HCC1588: 158
    HCC515: 5092
    HCC827: 4
    HCC95: 1218
    HCT116: 98
    HEC108: 1224
    HEC151: 188
    HEC1A: 1258
    HEC251: 1202
    HEC265: 1240
    HEK293: 1660
    HEK293T: 56
    HELA: 2642
    HEPG2: 2226
    HFL1: 628
    HIMG001: 30
    HIMG002: 30
    HL60: 92
    HME

Loading chunks: 100%|██████████| 16/16 [00:00<00:00, 22.24it/s, file=chunk_15.pkl]


Loaded and filtered 157138 samples from chunks
Computing biological context from 78569 control and 78569 perturb samples
Extracting graph-level latent representations...


Processing perturb samples: 100%|██████████| 78569/78569 [00:00<00:00, 100901.40it/s]


Computing mean representations...
References computed successfully.
Control ref shape: torch.Size([1, 100]), norm: 3.5154
perturb ref shape: torch.Size([1, 100]), norm: 3.5216
Biological context norm: 0.0070
ctrl latent shape : torch.Size([1, 100])
perturb latent shape: torch.Size([1, 100])


## 4. Predict perturbation responses on a query plate

In [4]:
from torch_geometric.data import DataLoader

tahoe_path = './data/minitahoe/p1_with_morgan.h5ad'
tahoe_dataset = ChunkedGeneGraphDataset(
    h5ad_paths=[tahoe_path], split='test',
    chunk_size=10000, auto_build_graph=True,
    species=9606, required_score=700,
)

loader = DataLoader(tahoe_dataset, batch_size=64, shuffle=False)
batch = next(iter(loader)).to(device)
predicted_exp, mask = model.predict(batch, latent_ctrl_ref, latent_ptrb_ref)
print('predicted expression shape:', predicted_exp.shape)

Found 4 chunks with 39546 total samples


/blue/qsong1/wang.qing/miniconda3/envs/info/lib/python3.11/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


predicted expression shape: torch.Size([64, 965, 1])


## 5. Evaluate on paired control–perturb samples

Re-uses the same evaluation pipeline that `src/main.py` runs in `--test_flag` mode:

* Filters Tahoe by `condition ∈ {control, perturb}`.
* Pairs them on `(celltype, main_ptrb, sub_ptrb)`.
* Computes per-sample Pearson / Spearman on absolute expression and on the DEG delta `pred − ctrl`.

Two equivalent ways to run it:

In [5]:
# Option A — call the helper directly from main.py
import argparse
from main import evaluate_perturbation_prediction

args = argparse.Namespace(
    device=device, batch_size=64,
    output_dir='./output', test_dataset_id=1,
)
results = evaluate_perturbation_prediction(
    model, tahoe_dataset, latent_ctrl_ref, latent_ptrb_ref,
    args, max_test_pairs=200, compute_deg=True,
)
print('absolute :', results['absolute_metrics'])
print('DEG delta:', results['deg_metrics'])

Filtering test dataset by condition...
Filtering dataset for condition: control


Filtering control samples: 100%|██████████| 39546/39546 [00:01<00:00, 33906.51it/s]


Found 19773 samples with condition 'control'
Filtering dataset for condition: perturb


Filtering perturb samples: 100%|██████████| 39546/39546 [00:01<00:00, 21689.06it/s]


Found 19773 samples with condition 'perturb'
Finding paired samples...
Found 19773 paired samples
Num of Test perturb:19773
Randomly selected 200 pairs for testing
Testing perturbation prediction on 200 paired samples...


Predicting perturbations: 100%|██████████| 4/4 [00:00<00:00, 16.37it/s]


Processing prediction results...
Processed 200 prediction pairs

Perturbation Prediction Metrics (Control → Predicted Stimulated vs Ground Truth Stimulated):
MSE                  40.841943
MAE                   6.379467
R2                 -391.892986
Pearson               0.295809
Spearman              0.158749
CosineSimilarity      0.410848
JS_Divergence         0.304281
SSIM                  0.010916
TopK_Overlap          0.000900
total_samples       200.000000
dtype: float64

Computing DEG (delta difference) metrics...

DEG Prediction Metrics (Predicted Delta vs Ground Truth Delta):
MSE                   40.842637
MAE                    6.379589
R2                 -1135.822248
Pearson                0.174971
Spearman               0.178482
CosineSimilarity       0.025448
JS_Divergence          0.004370
SSIM                  -0.002785
TopK_Overlap           0.047450
total_samples        200.000000
dtype: float64

✓ Results appended to: ./output/all_plates_results.csv
✓ Predictions sa

In [ ]:
# Option B — full sweep from the shell (one CSV row per plate)
# bash src/run_all.sh
# After it finishes, results land in:
#   output/all_plates_results.csv
#   output/predictions/plate_*_predictions.pkl

## 6. Put predictions on a common output scale

CellPert is trained on LINCS L1000 and evaluated on Mini Tahoe, and the two are on
different output scales: the LINCS training matrix has mean 8.43, the Mini Tahoe
targets have mean 0.14. Metrics react to that gap very differently. Pearson and
Spearman are invariant to any positive affine transformation of the prediction, so
they are unaffected. Mean squared error, R^2, cosine similarity and SSIM are not
invariant, and on raw output they largely measure the scale gap rather than the
prediction.

The diagnostic below removes the gap without touching the model. One scalar pair
`(a, b)` is fitted by least squares on a single plate,

```
a = Cov(P, G) / Var(P)      b = mean(G) - a * mean(P)
```

and then frozen and applied unchanged to every other plate, as `a * P + b`. It has
two degrees of freedom in total, so it cannot encode anything perturbation-specific;
it only places the output on the target scale.

Two things are worth checking whenever this is used. Pearson and Spearman must come
out identical before and after, which is a correctness check on the implementation.
And the fitted `a` must be reported: if `a` is near zero the calibrated prediction
collapses to the constant `b`, which equals the mean of the ground truth, and every
error-based metric then describes the ground truth alone rather than the model. On
the released checkpoint the fit gives `a = 0.657512` and `b = -4.083115`, and over
the held-out plates the metrics move as follows.

| metric | raw | calibrated |
| --- | --- | --- |
| Pearson | 0.3868 | 0.3868 |
| Spearman | 0.1380 | 0.1380 |
| MSE | 40.9121 | 0.1142 |
| R^2 | -555.63 | -0.0963 |
| Cosine similarity | 0.3825 | 0.4967 |
| SSIM | 0.0117 | 0.1934 |


In [ ]:
import pickle
import numpy as np

# Predictions written by src/main.py in --test_flag mode, or by src/run_all.sh.
with open('./output/predictions/plate_1_predictions.pkl', 'rb') as f:
    d = pickle.load(f)
P = np.asarray(d['predictions'], dtype=np.float64)    # cells x 965
G = np.asarray(d['ground_truth'], dtype=np.float64)

# Fit the two scalars on this plate, then treat them as fixed.
p, g = P.ravel(), G.ravel()
a = float(np.cov(p, g, bias=True)[0, 1] / p.var())
b = float(g.mean() - a * p.mean())
print('fitted on plate 1:  a = %.6f   b = %.6f' % (a, b))
if abs(a) < 1e-3:
    print('warning: a is near zero, the calibrated output is essentially the constant b')

Q = a * P + b        # apply to this plate, and to every other plate unchanged


def report(name, X):
    xc = X - X.mean(1, keepdims=True)
    gc = G - G.mean(1, keepdims=True)
    den = np.linalg.norm(xc, axis=1) * np.linalg.norm(gc, axis=1)
    pearson = np.nanmean(np.where(den > 0, (xc * gc).sum(1) / np.maximum(den, 1e-12), np.nan))
    cosine = np.nanmean((X * G).sum(1) /
                        (np.linalg.norm(X, axis=1) * np.linalg.norm(G, axis=1) + 1e-12))
    mse = float(((X - G) ** 2).mean())
    r2 = float(1 - ((X - G) ** 2).sum() / ((G - G.mean()) ** 2).sum())
    print('%-12s MSE %10.4f   R2 %10.4f   Pearson %.4f   cosine %.4f'
          % (name, mse, r2, pearson, cosine))


report('raw', P)
report('calibrated', Q)

# Pearson is identical on both lines. That is the point: the affine changes the scale
# of the output and nothing about the ordering or the shape of what the model predicts.


## 7. Latent-space alignment: Figure 3b and Supplementary Figure S1

Both figures rest on one object, the encoder's graph-level mean `graph_mu`. Figure 3b
measures how separable the two datasets are before and after the encoder. Supplementary
Figure S1 shows the same thing as a picture.

The measurement needs a control, and that is the reason for the second and third
colourings below. A representation that simply discarded variation would lower the
dataset separability just as alignment would. So the same statistics are also computed
for cell line and for perturbation state within Mini Tahoe. Alignment should reduce the
first while leaving the other two intact; collapse would flatten all three.

`umap-learn` is needed for the last cell and is not part of `environment.yml`; install
it with `pip install umap-learn` if it is missing.


In [ ]:
import numpy as np
import torch
from torch_geometric.loader import DataLoader as GeoLoader

# Sizes kept small so the section runs in a few minutes. The published analysis used
# 15000 LINCS cells and 2500 from each plate.
N_LINCS, N_TAHOE, BATCH = 2000, 2000, 256
rng = np.random.default_rng(0)


def graph_mu(dataset, n, seed=0):
    """Encoder graph-level mean for a random subsample. No reference context needed."""
    r = np.random.default_rng(seed)
    idx = r.choice(len(dataset), size=min(n, len(dataset)), replace=False)
    out = []
    model.eval()
    with torch.no_grad():
        for s in range(0, len(idx), BATCH):
            chunk = [dataset[int(i)] for i in idx[s:s + BATCH]]
            b = next(iter(GeoLoader(chunk, batch_size=len(chunk), shuffle=False))).to(device)
            _, _, gmu, _ = model.encoder(b.x, b.edge_index, b.batch)
            out.append(gmu.cpu().numpy())
    return np.concatenate(out).astype(np.float32), idx


import scanpy as sc

lincs_ad = sc.read_h5ad(lincs_path)
tahoe_ad = sc.read_h5ad(tahoe_path)
to_dense = lambda X: np.asarray(X.todense() if hasattr(X, 'todense') else X, dtype=np.float32)

LZ, li = graph_mu(lincs_dataset, N_LINCS, seed=0)
TZ, ti = graph_mu(tahoe_dataset, N_TAHOE, seed=1)
LX = to_dense(lincs_ad.X)[li]
TX = to_dense(tahoe_ad.X)[ti]

# Labels for the three colourings.
dataset_lab = np.array(['LINCS'] * len(LX) + ['Mini-Tahoe'] * len(TX))
cell_line = tahoe_ad.obs['cell_type'].astype(str).values[ti]
condition = tahoe_ad.obs['condition'].astype(str).values[ti]

print('raw   ', LX.shape, TX.shape)
print('latent', LZ.shape, TZ.shape)
print('cell lines in this Tahoe sample:', len(set(cell_line)))


### 7a. The quantities plotted in Figure 3b

Two statistics, each computed once in the raw 965-gene space and once in the
100-dimensional latent space. Lower means the two datasets are harder to tell apart,
which is what alignment should achieve.

* **KS statistic**, the two-sample Kolmogorov-Smirnov statistic between LINCS and Mini
  Tahoe, computed per dimension and averaged over dimensions.
* **Silhouette score** with `metric='cosine'`, labelling each cell by its dataset.

For reference, on plate 1 the published run gives KS 0.9988 in raw space against 0.9814
in the latent, and silhouette 0.9092 against 0.8786. Over the fourteen plates the means
are 0.9996 against 0.9800 and 0.9335 against 0.9134; KS falls on all fourteen plates and
silhouette on ten of them.


In [ ]:
from scipy.stats import ks_2samp
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt


def ks_mean(A, B):
    """Mean over dimensions of the two-sample KS statistic."""
    return float(np.mean([ks_2samp(A[:, j], B[:, j])[0] for j in range(A.shape[1])]))


def sil(A, B, labels, n=4000, seed=0):
    M = np.concatenate([A, B])
    r = np.random.default_rng(seed)
    i = r.choice(len(M), size=min(n, len(M)), replace=False)
    return float(silhouette_score(M[i], np.asarray(labels)[i], metric='cosine'))


rows = []
for space, (A, B) in (('raw', (LX, TX)), ('latent', (LZ, TZ))):
    rows.append((space, ks_mean(A, B), sil(A, B, dataset_lab)))
    print('%-7s  KS %.4f   silhouette %.4f' % rows[-1])

# The positive control: the same silhouette for cell line and for perturbation state,
# inside Mini Tahoe only. Alignment should leave these two alone.
for name, lab in (('cell line', cell_line), ('perturbation state', condition)):
    a = silhouette_score(TX, lab, metric='cosine')
    b = silhouette_score(TZ, lab, metric='cosine')
    print('%-20s raw %+.4f  latent %+.4f' % (name, a, b))

# Figure 3b is this comparison drawn as one dumbbell per plate.
fig, ax = plt.subplots(figsize=(4.2, 3.4))
for k, (metric, color) in enumerate((('KS statistic', '#E64B35'),
                                     ('Silhouette', '#4DBBD5'))):
    lo, hi = rows[1][k + 1], rows[0][k + 1]
    ax.plot([k, k], [lo, hi], color=color, lw=2.5, zorder=1)
    ax.scatter([k], [hi], s=70, facecolor=color, edgecolor='black', zorder=2,
               label='Raw space' if k == 0 else None)
    ax.scatter([k], [lo], s=70, marker='^', facecolor=color, edgecolor='black',
               zorder=2, label='Latent space' if k == 0 else None)
ax.set_xticks([0, 1]); ax.set_xticklabels(['KS statistic', 'Silhouette'])
ax.set_ylabel('Score (lower = better alignment)')
ax.set_title('This plate, raw against latent')
ax.legend(frameon=False, fontsize=8)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.show()


### 7b. Supplementary Figure S1

The same two spaces drawn with UMAP, each coloured three ways. The top row is the raw
965-gene space, the bottom row the latent space. In the raw space the two datasets
occupy separate regions; in the latent space they overlap, while the Mini Tahoe cell
lines stay distinguishable, which is the difference between alignment and collapse.


In [ ]:
import umap
import matplotlib.pyplot as plt

emb = {}
for space, (A, B) in (('raw', (LX, TX)), ('latent', (LZ, TZ))):
    emb[space] = umap.UMAP(n_neighbors=15, min_dist=0.1,
                           random_state=0).fit_transform(np.concatenate([A, B]))

GREY = '#DCDCDC'
n_l = len(LX)
lines = sorted(set(cell_line))
line_color = {c: plt.get_cmap('tab20')(i % 20) for i, c in enumerate(lines)}

fig, axes = plt.subplots(2, 3, figsize=(13.5, 8.6))
for row, space in enumerate(('raw', 'latent')):
    E = emb[space]
    title = ('Raw expression space (965 genes)' if space == 'raw'
             else 'CellPert latent space (100 dimensions)')

    ax = axes[row, 0]
    for k, c in (('LINCS', '#1F77B4'), ('Mini-Tahoe', '#D62728')):
        m = dataset_lab == k
        ax.scatter(E[m, 0], E[m, 1], s=5, c=c, linewidths=0, label=k)
    ax.legend(fontsize=8, markerscale=2.4, frameon=True)
    ax.set_title(title + '\nby dataset', fontsize=10)

    ax = axes[row, 1]
    ax.scatter(E[:n_l, 0], E[:n_l, 1], s=5, c=GREY, linewidths=0, zorder=1)
    for c in lines:
        m = np.zeros(len(E), bool); m[n_l:] = cell_line == c
        ax.scatter(E[m, 0], E[m, 1], s=5, color=line_color[c], linewidths=0, zorder=2)
    ax.set_title(title + '\nby cell line (Mini-Tahoe)', fontsize=10)

    ax = axes[row, 2]
    ax.scatter(E[:n_l, 0], E[:n_l, 1], s=5, c=GREY, linewidths=0, label='LINCS', zorder=1)
    for k, c in (('control', '#2CA02C'), ('perturb', '#9467BD')):
        m = np.zeros(len(E), bool); m[n_l:] = condition == k
        ax.scatter(E[m, 0], E[m, 1], s=5, c=c, linewidths=0, label=k, zorder=2)
    ax.legend(fontsize=8, markerscale=2.4, frameon=True)
    ax.set_title(title + '\nby perturbation state', fontsize=10)

for ax in axes.ravel():
    ax.set_xlabel('UMAP1'); ax.set_ylabel('UMAP2')
plt.tight_layout(); plt.show()
